# 5. 2026 salary holdout

Train using target seasons through 2025 and evaluate only on 2026 development rows. Profile forecasts for holdout rows come from a forecaster trained only on pre-holdout outcomes. Historical comparison players and the PCA basis are also restricted to pre-holdout outcomes.

This is a temporal backtest, not pristine independent confirmation: the project has previously inspected the development data, and profile-model settings were selected earlier using development results. The historical test split is not loaded.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import sys
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib

try:
    from IPython.display import display
except ImportError:
    display = print

rootPath = Path.cwd()
while not (rootPath / 'data' / 'processed').exists() and rootPath != rootPath.parent:
    rootPath = rootPath.parent
if not (rootPath / 'data' / 'processed').exists(): raise FileNotFoundError('Open this notebook from inside the project.')
sys.path.insert(0, str(rootPath))


processedPath = rootPath / 'data' / 'processed'
resultRoot = rootPath / 'results' / 'development' / 'multitask_profile'
modelRoot = rootPath / 'models' / 'numpy_salary_ridge'
holdoutStartSeason = 2026
selectedDropout = 0.1
selectedEpochs = 66
learningRate = 0.001
weightDecay = 0.0001
batchSize = 64
print('Holdout target seasons: ' + str(holdoutStartSeason) + ' and later')

## Shared network and profile-target implementations

These implementations are included directly so this workflow does not import project Python modules.


In [ ]:
SKILL_TARGETS = [
    "pie",
    "ws_48",
    "bpm",
    "ts_percent",
    "usg_percent",
    "trb_percent",
    "ast_percent",
    "stl_percent",
    "blk_percent",
    "tov_percent",
    "x3p_ar",
    "f_tr",
    "e_fg_percent",
    "x3p_percent",
    "ft_percent",
]
PROFILE_TARGETS = SKILL_TARGETS + ["minutes"]
PCA_TARGETS = [
    target for target in SKILL_TARGETS
    if target not in {"x3p_percent", "ft_percent"}
]
NEXT_SEASON_COLUMNS = {
    target: ("next_season_minutes" if target == "minutes" else "next_season_" + target)
    for target in PROFILE_TARGETS
}
BASELINE_COLUMNS = {
    target: ("mp" if target == "minutes" else target)
    for target in PROFILE_TARGETS
}

In [ ]:
def initialize_parameters(layer_dims, seed=42):
    """Initialize a ReLU MLP with a multi-output linear final layer."""
    if len(layer_dims) < 2 or any(size < 1 for size in layer_dims):
        raise ValueError("Layer dimensions must be positive and include input/output sizes.")
    rng = np.random.default_rng(seed)
    parameters = {}
    for layer in range(1, len(layer_dims)):
        scale = np.sqrt(2.0 / layer_dims[layer - 1])
        parameters[f"W{layer}"] = (
            rng.standard_normal((layer_dims[layer], layer_dims[layer - 1])) * scale
        ).astype(np.float32)
        parameters[f"b{layer}"] = np.zeros((layer_dims[layer], 1), dtype=np.float32)
    return parameters

In [ ]:
def forward(inputs, parameters, keep_prob=1.0, training=False, rng=None):
    """Forward pass; input shape is features-by-examples."""
    if not 0 < keep_prob <= 1:
        raise ValueError("keep_prob must be in (0, 1].")
    if training and keep_prob < 1 and rng is None:
        raise ValueError("A random generator is required when dropout is enabled.")
    activations = inputs
    hidden_caches = []
    n_layers = len(parameters) // 2
    for layer in range(1, n_layers):
        previous = activations
        z = parameters[f"W{layer}"] @ previous + parameters[f"b{layer}"]
        activations = np.maximum(z, 0)
        dropout_mask = None
        if training and keep_prob < 1:
            dropout_mask = (
                (rng.random(activations.shape) < keep_prob).astype(np.float32)
                / keep_prob
            )
            activations = activations * dropout_mask
        hidden_caches.append((previous, z, dropout_mask))
    output_layer = n_layers
    output_input = activations
    predictions = (
        parameters[f"W{output_layer}"] @ output_input
        + parameters[f"b{output_layer}"]
    )
    return predictions, (hidden_caches, output_input)

In [ ]:
def masked_mse(predictions, targets, mask):
    """Mean of per-target MSEs, each averaged over its observed labels."""
    mask = np.asarray(mask, dtype=bool)
    counts = mask.sum(axis=1)
    valid_targets = counts > 0
    if not valid_targets.any():
        raise ValueError("Batch has no observed targets.")
    per_target = np.zeros(predictions.shape[0], dtype=np.float64)
    for target in np.flatnonzero(valid_targets):
        errors = predictions[target, mask[target]] - targets[target, mask[target]]
        per_target[target] = np.mean(errors.astype(np.float64) ** 2)
    return float(per_target[valid_targets].mean()), per_target, counts

In [ ]:
def backward(predictions, targets, mask, caches, parameters, weight_decay=0.0):
    """Backpropagate a task-balanced masked MSE and coupled L2 penalty."""
    hidden_caches, output_input = caches
    n_layers = len(parameters) // 2
    counts = mask.sum(axis=1).astype(np.float32)
    n_valid_targets = max(int(np.count_nonzero(counts)), 1)
    safe_counts = np.maximum(counts, 1.0)
    d_z = (
        2.0
        * (predictions - targets)
        * mask
        / safe_counts[:, None]
        / n_valid_targets
    )
    gradients = {}
    gradients[f"dW{n_layers}"] = (
        d_z @ output_input.T + weight_decay * parameters[f"W{n_layers}"]
    )
    gradients[f"db{n_layers}"] = (
        d_z.sum(axis=1, keepdims=True) + weight_decay * parameters[f"b{n_layers}"]
    )
    d_a = parameters[f"W{n_layers}"].T @ d_z

    for layer in reversed(range(1, n_layers)):
        previous, z, dropout_mask = hidden_caches[layer - 1]
        if dropout_mask is not None:
            d_a = d_a * dropout_mask
        d_z = d_a * (z > 0)
        gradients[f"dW{layer}"] = (
            d_z @ previous.T + weight_decay * parameters[f"W{layer}"]
        )
        gradients[f"db{layer}"] = (
            d_z.sum(axis=1, keepdims=True) + weight_decay * parameters[f"b{layer}"]
        )
        d_a = parameters[f"W{layer}"].T @ d_z
    return gradients

In [ ]:
def adam_initialize(parameters):
    first = {key: np.zeros_like(value) for key, value in parameters.items()}
    second = {key: np.zeros_like(value) for key, value in parameters.items()}
    return first, second

In [ ]:
def adam_update(parameters, gradients, first, second, step, learning_rate,
                beta1=0.9, beta2=0.999, epsilon=1e-8):
    for key in parameters:
        first[key] = beta1 * first[key] + (1.0 - beta1) * gradients["d" + key]
        second[key] = beta2 * second[key] + (1.0 - beta2) * gradients["d" + key] ** 2
        first_hat = first[key] / (1.0 - beta1**step)
        second_hat = second[key] / (1.0 - beta2**step)
        parameters[key] -= learning_rate * first_hat / (np.sqrt(second_hat) + epsilon)
    return parameters, first, second

In [ ]:
def predict(inputs, parameters, target_means, target_scales, minute_index=None):
    predictions, _ = forward(inputs, parameters)
    decoded = predictions.astype(np.float64).T * target_scales + target_means
    if minute_index is not None:
        decoded[:, minute_index] = np.maximum(decoded[:, minute_index], 0.0)
    return decoded

## Load only train and development splits

All 2026 development rows are held out. Training rows after the cutoff are excluded from both profile forecaster fitting and salary-model fitting; the salary model uses target seasons through 2025.

In [ ]:
with np.load(processedPath / 'model_data_rates.npz', allow_pickle=False) as modelData:
    xTrain = modelData['X_train'].astype(np.float32)
    xDev = modelData['X_dev'].astype(np.float32)
    yTrainProfile = modelData['profile_y_train'].astype(np.float64)
    trainProfileMask = modelData['profile_mask_train'].astype(bool)
    featureNames = modelData['feature_names'].astype(str).tolist()
    targetNames = modelData['profile_target_names'].astype(str).tolist()

trainRows = pd.read_csv(processedPath / 'train_rows_rates.csv')
devRows = pd.read_csv(processedPath / 'dev_rows_rates.csv')
salaryPairs = pd.read_csv(processedPath / 'salary_cap_normalized_pairs.csv')
salaryColumns = ['player_id','target_season','salary_cap_share','salary_available_check','salary_needs_source_review','salary_cap_usd']
salaryFeatureRows = salaryPairs[salaryColumns].rename(columns={'salary_available_check':'model_salary_available','salary_needs_source_review':'model_salary_needs_review'})
trainRows = trainRows.merge(salaryFeatureRows, on=['player_id','target_season'], how='left', validate='one_to_one')
devRows = devRows.merge(salaryFeatureRows, on=['player_id','target_season'], how='left', validate='one_to_one')

if len(trainRows) != len(xTrain) or len(devRows) != len(xDev): raise ValueError('Player rows do not align with model arrays.')
if targetNames != PROFILE_TARGETS: raise ValueError('Profile target order differs from the shared model.')
if not trainRows[['player_id','season','target_season']].equals(pd.read_csv(processedPath / 'train_rows_rates.csv')[['player_id','season','target_season']]): raise ValueError('Training row order changed during the salary join.')

reliableTrainMask = trainRows.model_salary_available.fillna(False).to_numpy() & ~trainRows.model_salary_needs_review.fillna(True).to_numpy() & trainRows.salary_cap_share.gt(0).fillna(False).to_numpy()
reliableDevMask = devRows.model_salary_available.fillna(False).to_numpy() & ~devRows.model_salary_needs_review.fillna(True).to_numpy() & devRows.salary_cap_share.gt(0).fillna(False).to_numpy()
earlyTrainMask = trainRows.target_season.lt(holdoutStartSeason).to_numpy()
holdoutDevMask = devRows.target_season.ge(holdoutStartSeason).to_numpy()
print('Earlier training rows: ' + str(int(earlyTrainMask.sum())))
print('Later development rows: ' + str(int(holdoutDevMask.sum())))
print('Reliable earlier salary labels: ' + str(int((reliableTrainMask & earlyTrainMask).sum())))
print('Reliable holdout salary labels: ' + str(int((reliableDevMask & holdoutDevMask).sum())))
display(devRows.loc[holdoutDevMask].groupby('target_season').agg(rows=('player_id','size'), reliable_salary=('model_salary_available','sum')).reset_index())

## Profile-network training helper

Use the established shared-network architecture and settings, but fit each forecast network only on target-season outcomes earlier than the rows it forecasts.

In [ ]:
hiddenWidths = [64, 32]
layerWidths = [xTrain.shape[1], *hiddenWidths, len(PROFILE_TARGETS)]

def fitProfileNetwork(fitIndices, randomSeed):
    fitInputs = xTrain[fitIndices]
    fitValues = yTrainProfile[fitIndices]
    fitMask = trainProfileMask[fitIndices]
    targetMeans = np.zeros(len(PROFILE_TARGETS), dtype=float)
    targetScales = np.ones(len(PROFILE_TARGETS), dtype=float)
    scaledTargets = np.zeros(fitValues.shape, dtype=np.float32)

    for targetIndex, targetName in enumerate(PROFILE_TARGETS):
        observedValues = fitValues[fitMask[:, targetIndex], targetIndex]
        if not len(observedValues): raise ValueError('No fit labels for ' + targetName)
        targetMeans[targetIndex] = observedValues.mean()
        targetScales[targetIndex] = observedValues.std()
        if targetScales[targetIndex] <= 0: raise ValueError('No target variation for ' + targetName)
        targetRows = fitMask[:, targetIndex]
        scaledTargets[targetRows, targetIndex] = ((fitValues[targetRows, targetIndex] - targetMeans[targetIndex]) / targetScales[targetIndex]).astype(np.float32)

    parameters = initialize_parameters(layerWidths, seed=randomSeed)
    firstMoment, secondMoment = adam_initialize(parameters)
    shuffleRandom = np.random.default_rng(randomSeed + 10)
    dropoutRandom = np.random.default_rng(randomSeed + 20)
    updateStep = 0

    for epochNumber in range(selectedEpochs):
        batchOrder = shuffleRandom.permutation(len(fitIndices))
        for batchStart in range(0, len(batchOrder), batchSize):
            batchIndices = batchOrder[batchStart:batchStart + batchSize]
            batchOutput, cache = forward(fitInputs[batchIndices].T, parameters, keep_prob=1 - selectedDropout, training=True, rng=dropoutRandom)
            gradients = backward(batchOutput, scaledTargets[batchIndices].T, fitMask[batchIndices].T.astype(np.float32), cache, parameters, weightDecay)
            updateStep += 1
            parameters, firstMoment, secondMoment = adam_update(parameters, gradients, firstMoment, secondMoment, updateStep, learningRate)

    return parameters, targetMeans, targetScales

def buildSalaryFeatures(playerRows, forecastRows):
    basicFeatures = playerRows[['age','pos','g','gs','mp','mp_per_game','target_season']].copy()
    basicFeatures['pos'] = basicFeatures.pos.fillna('Unknown').astype(str)
    forecastFeatures = pd.DataFrame(forecastRows, columns=PROFILE_TARGETS, index=basicFeatures.index)
    forecastFeatures = forecastFeatures.rename(columns={target:'forecast_' + target for target in PROFILE_TARGETS})
    return pd.concat([basicFeatures, forecastFeatures], axis=1)

numericColumns = ['age','g','gs','mp','mp_per_game','target_season'] + ['forecast_' + target for target in PROFILE_TARGETS]
categoricalColumns = ['pos']
featureTransform = ColumnTransformer([
    ('numeric', StandardScaler(), numericColumns),
    ('position', OneHotEncoder(handle_unknown='ignore'), categoricalColumns),
])

## Create time-causal forecasts for salary-model training

For each block of historical training seasons, fit the profile network only on earlier target seasons, then forecast the block. The earliest seasons are left out until enough history exists.

In [ ]:
historicalYears = np.sort(trainRows.loc[earlyTrainMask, 'target_season'].unique())
forecastYears = historicalYears[5:]
yearBlocks = [block for block in np.array_split(forecastYears, 5) if len(block)]
historicalForecasts = np.full((len(trainRows), len(PROFILE_TARGETS)), np.nan, dtype=float)

for blockNumber, yearBlock in enumerate(yearBlocks):
    blockStart = int(yearBlock.min())
    fitIndices = np.flatnonzero(trainRows.target_season.lt(blockStart).to_numpy())
    forecastIndices = np.flatnonzero(trainRows.target_season.isin(yearBlock).to_numpy())
    if not len(fitIndices) or not len(forecastIndices): continue
    parameters, targetMeans, targetScales = fitProfileNetwork(fitIndices, 200 + blockNumber)
    historicalForecasts[forecastIndices] = predict(xTrain[forecastIndices].T, parameters, targetMeans, targetScales, minute_index=PROFILE_TARGETS.index('minutes'))
    print('Forecasted seasons ' + str(int(yearBlock.min())) + '-' + str(int(yearBlock.max())) + ' from earlier seasons only.')

forecastReadyMask = np.isfinite(historicalForecasts).all(axis=1)
salaryFitMask = reliableTrainMask & earlyTrainMask & forecastReadyMask
print('Time-causal salary training rows: ' + str(int(salaryFitMask.sum())))
if salaryFitMask.sum() < 1000: raise ValueError('Too few earlier salary rows for the temporal backtest.')

## Fit the salary model and holdout forecaster

The salary model sees only reliable pre-cutoff salary labels. A separate profile network fit on the same pre-cutoff era creates production forecasts for later development rows.

In [ ]:
trainSalaryFeatures = buildSalaryFeatures(trainRows, historicalForecasts)
salaryModel = Pipeline([
    ('features', featureTransform),
    ('ridge', Ridge(alpha=10.0)),
])
salaryModel.fit(trainSalaryFeatures.loc[salaryFitMask], np.log1p(trainRows.loc[salaryFitMask, 'salary_cap_share'].to_numpy(dtype=float)))

earlyProfileIndices = np.flatnonzero(earlyTrainMask)
holdoutProfileModel, holdoutTargetMeans, holdoutTargetScales = fitProfileNetwork(earlyProfileIndices, 42)
holdoutDevIndices = np.flatnonzero(holdoutDevMask)
holdoutForecasts = predict(xDev[holdoutDevIndices].T, holdoutProfileModel, holdoutTargetMeans, holdoutTargetScales, minute_index=PROFILE_TARGETS.index('minutes'))
holdoutSalaryFeatures = buildSalaryFeatures(devRows.iloc[holdoutDevIndices].reset_index(drop=True), holdoutForecasts)
holdoutModelShares = np.maximum(0, np.expm1(salaryModel.predict(holdoutSalaryFeatures)))
print('Salary-model fit rows: ' + str(int(salaryFitMask.sum())))
print('Holdout profile forecast rows: ' + str(len(holdoutDevIndices)))

## Rebuild the historical-comps baseline

Fit feature scaling and PCA only on complete pre-cutoff training outcomes. For each holdout player, search only historical profiles whose outcome season was known by that player's prior season.

In [ ]:
skillColumns = ['next_season_' + target for target in PCA_TARGETS]
trainSkillValues = trainRows[skillColumns].to_numpy(dtype=float)
completeSkillMask = np.isfinite(trainSkillValues).all(axis=1) & earlyTrainMask
skillScaler = StandardScaler()
scaledTrainingSkills = skillScaler.fit_transform(trainSkillValues[completeSkillMask])
profilePca = PCA(n_components=0.95, svd_solver='full')
trainingCoordinates = profilePca.fit_transform(scaledTrainingSkills)
candidateRows = trainRows.loc[completeSkillMask].reset_index(drop=True)
candidateSalaryShares = candidateRows.salary_cap_share.to_numpy(dtype=float)
candidateSalaryFlags = candidateRows.model_salary_needs_review.fillna(True).to_numpy()
holdoutSkillIndices = [PROFILE_TARGETS.index(target) for target in PCA_TARGETS]
holdoutCoordinates = profilePca.transform(skillScaler.transform(holdoutForecasts[:, holdoutSkillIndices]))
holdoutRows = devRows.iloc[holdoutDevIndices].reset_index(drop=True)
compSummaries = []

for queryIndex, queryRow in holdoutRows.iterrows():
    eligibleMask = candidateRows.target_season.le(queryRow.season) & candidateRows.player_id.ne(queryRow.player_id)
    eligibleIndices = np.flatnonzero(eligibleMask.to_numpy())
    if not len(eligibleIndices): continue
    distances = np.linalg.norm(trainingCoordinates[eligibleIndices] - holdoutCoordinates[queryIndex], axis=1)
    rankedIndices = eligibleIndices[np.argsort(distances, kind='stable')]
    selectedIndices = []
    seenPlayers = set()
    for candidateIndex in rankedIndices:
        playerId = candidateRows.iloc[candidateIndex].player_id
        if playerId in seenPlayers: continue
        selectedIndices.append(candidateIndex)
        seenPlayers.add(playerId)
        if len(selectedIndices) == 10: break
    usableSalary = [candidateSalaryShares[index] for index in selectedIndices if np.isfinite(candidateSalaryShares[index]) and candidateSalaryShares[index] > 0 and not candidateSalaryFlags[index]]
    compSummaries.append({
        'query_player':queryRow.player,
        'query_player_id':queryRow.player_id,
        'query_season':int(queryRow.season),
        'query_target_season':int(queryRow.target_season),
        'usable_salary_comp_count':len(usableSalary),
        'comp_median_share':float(np.median(usableSalary)) if len(usableSalary) >= 5 else np.nan,
        'comp_names':'; '.join(candidateRows.iloc[index].player for index in selectedIndices),
    })

holdoutComps = pd.DataFrame(compSummaries)
print('PCA components fit on early training profiles: ' + str(profilePca.n_components_))
print('Holdout rows with comp summaries: ' + str(len(holdoutComps)))

## Score the temporal holdout

Compare the Ridge estimate with the rebuilt comp-median baseline on reliable later-season development salaries. No model setting is selected using these holdout scores.

In [ ]:
holdoutResults = holdoutRows[['player','player_id','season','target_season','salary_cap_share','salary_cap_usd','model_salary_available','model_salary_needs_review']].copy()
holdoutResults['salaryModelShare'] = holdoutModelShares
holdoutResults = holdoutResults.merge(holdoutComps[['query_player_id','query_target_season','usable_salary_comp_count','comp_median_share','comp_names']], left_on=['player_id','target_season'], right_on=['query_player_id','query_target_season'], how='left', validate='one_to_one')
holdoutResults['compMedianShare'] = holdoutResults.comp_median_share
holdoutResults['salaryModelDollars'] = holdoutResults.salaryModelShare * holdoutResults.salary_cap_usd
holdoutResults['compMedianDollars'] = holdoutResults.compMedianShare * holdoutResults.salary_cap_usd
scoreMask = holdoutResults.model_salary_available.fillna(False) & ~holdoutResults.model_salary_needs_review.fillna(True) & holdoutResults.compMedianShare.notna()
if not scoreMask.any(): raise ValueError('No reliable chronological holdout rows with enough usable comps.')

def scoreSalaryEstimate(actualShares, estimatedShares):
    errors = (estimatedShares - actualShares) * 100
    return {
        'rows':int(len(errors)),
        'mae_cap_percentage_points':float(np.mean(np.abs(errors))),
        'median_absolute_error_cap_points':float(np.median(np.abs(errors))),
        'rmse_cap_percentage_points':float(np.sqrt(np.mean(errors ** 2))),
    }

actualShares = holdoutResults.loc[scoreMask, 'salary_cap_share'].to_numpy(dtype=float)
salaryModelShares = holdoutResults.loc[scoreMask, 'salaryModelShare'].to_numpy(dtype=float)
compMedianShares = holdoutResults.loc[scoreMask, 'compMedianShare'].to_numpy(dtype=float)
holdoutMetrics = pd.DataFrame([
    {'method':'Ridge salary model', **scoreSalaryEstimate(actualShares, salaryModelShares)},
    {'method':'Historical-comp median', **scoreSalaryEstimate(actualShares, compMedianShares)},
])
display(holdoutMetrics.round(3))
print('Target-season holdout: ' + str(holdoutStartSeason) + ' only')
print('Scored holdout rows: ' + str(int(scoreMask.sum())))
print('Test split loaded: False')

## Save the temporal model and predictions

Keep this run separate from the development-fitted salary model.

In [ ]:
runId = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
modelFolder = modelRoot / ('temporal_' + runId)
modelFolder.mkdir(parents=True, exist_ok=False)
joblib.dump(salaryModel, modelFolder / 'salary_ridge.joblib')
modelInfo = {
    'model':'Ridge regression on log salary-cap share',
    'target':'next-season salary-cap share',
    'holdout_target_season':holdoutStartSeason,
    'training_rows':int(salaryFitMask.sum()),
    'holdout_scored_rows':int(scoreMask.sum()),
    'profile_forecast_policy':'time-causal expanding windows for training rows; pre-holdout-only profile network for holdout forecasts',
    'pca_fit_policy':'complete early-training skill profiles only',
    'excluded_features':['hometown','prior salary','player ID','team'],
    'review_salary_policy':'unavailable and source-review-flagged salary labels excluded',
    'development_history_note':'Temporal backtest is not pristine independent confirmation; earlier development data and profile settings were previously examined.',
    'test_split_loaded':False,
    'metrics':holdoutMetrics.to_dict(orient='records'),
}
(modelFolder / 'metadata.json').write_text(json.dumps(modelInfo, indent=2) + '\n', encoding='utf-8')

latestResultFolders = [folder for folder in resultRoot.iterdir() if folder.is_dir() and (folder / 'development_report.json').exists()]
latestResultFolder = max(latestResultFolders, key=lambda folder: folder.stat().st_mtime)
holdoutResults.to_csv(latestResultFolder / 'temporal_salary_holdout_predictions.csv', index=False)
holdoutMetrics.to_csv(latestResultFolder / 'temporal_salary_holdout_metrics.csv', index=False)
print('Saved temporal salary model: ' + str(modelFolder))
print('Saved holdout results: ' + str(latestResultFolder / 'temporal_salary_holdout_predictions.csv'))

exampleResults = holdoutResults.loc[scoreMask].copy()
exampleResults['modelAbsoluteError'] = (exampleResults.salaryModelShare - exampleResults.salary_cap_share).abs()
exampleResults['compAbsoluteError'] = (exampleResults.compMedianShare - exampleResults.salary_cap_share).abs()
display(exampleResults.sort_values('target_season').head(12)[['player','target_season','salary_cap_share','salaryModelShare','compMedianShare','usable_salary_comp_count','comp_names']])